# Remote CLIP Colab Worker — Control Panel

Runs the **Remote CLIP Colab** inference worker on this runtime and exposes a
control panel for it. Your local ComfyUI connects with the printed
`base_url` + `auth_token` via the `Load Remote CLIP (Colab)` node.

**Cells**
1. Setup — fetch the runtime (git clone / uploaded folder / Google Drive) and install deps
2. Configure — token, tunnel, backend, startup models
3. Launch — start the worker in the background, wait for the public URL
4. Download — fetch text-encoder / LoRA files into `models/` (user-initiated)
5. **Control Panel** — status / load / switch / unload / LoRAs / cache / shutdown / encode test
6. Log — tail the worker log
7. Stop — shut the worker down

Runtime type: **GPU (T4/L4/A100)** recommended — the native backend (ComfyUI's own
text-encoder stack) activates automatically. On **TPU** the worker falls back to
the pure-transformers backend. NVFP4/AWQ checkpoints (e.g. MiniMax H3) need an
Ampere-or-newer GPU.

In [ ]:
# @title 1 · Setup runtime { display-mode: "form" }
import os, subprocess, sys

REPO_URL = ""  # @param {type:"string"} — set to clone; leave empty if the colab/ folder is already here or on Drive
RUNTIME_DIR = "/content/ComfyUI-RemoteCLIPColab/colab"  # @param {type:"string"} — e.g. /content/ComfyUI-RemoteCLIPColab/colab or a Drive path

if REPO_URL.strip() and not os.path.isdir(RUNTIME_DIR):
    repo_root = os.path.dirname(RUNTIME_DIR.rstrip("/")) or "/content/ComfyUI-RemoteCLIPColab"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL.strip(), repo_root], check=True)

assert os.path.isfile(os.path.join(RUNTIME_DIR, "worker.py")), (
    f"worker.py not found under {RUNTIME_DIR}.\n"
    "Set REPO_URL to your fork, or upload the colab/ folder (File → Upload), "
    "or mount Drive and point RUNTIME_DIR at it.")
os.chdir(RUNTIME_DIR)

def _is_tpu():
    try:
        import torch_xla  # noqa: F401
        return True
    except ImportError:
        return False

print("installing worker dependencies (first run takes a couple of minutes) ...")
if _is_tpu():
    # TPU images pin torch+torch_xla as a pair; reinstalling torch breaks XLA.
    pkgs = [l.split("#", 1)[0].strip() for l in open("requirements.txt")]
    pkgs = [p for p in pkgs if p and not p.startswith(("torch", "comfy-"))]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
    print("TPU runtime: kept preinstalled torch/torch_xla, skipped comfy-kitchen/aimdo (native backend unused on TPU)")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # optional GPU inference-acceleration kernels for the native backend
    # (cell 2 ATTENTION=sage/flash); sdpa needs no extra package
    for pkg in ("sageattention",):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                           capture_output=True, text=True)
        print(pkg, "installed" if r.returncode == 0 else "unavailable (ATTENTION=sage will be rejected)")
print("runtime ready:", os.getcwd())

In [ ]:
# @title 2 · Configure { display-mode: "form" }
AUTH_TOKEN = ""  # @param {type:"string"} — leave empty to auto-generate
TUNNEL = "cloudflare"  # @param ["cloudflare", "ngrok", "direct"]
NGROK_TOKEN = ""  # @param {type:"string"} — only for TUNNEL=ngrok
PORT = 8199  # @param {type:"integer"}
ENGINE = "auto"  # @param ["auto", "native", "hf"] — native = ComfyUI's own stack (recommended on GPU)
DEVICE = "auto"  # @param ["auto", "cuda", "tpu", "cpu"]
ATTENTION = "auto"  # @param ["auto", "sdpa", "sage", "flash"] — GPU inference acceleration
ATTENTION_HF = "sdpa"  # @param ["sdpa", "eager", "flash_attention_2"] — hf-backend kernel (CUDA)
XLA_CACHE = True  # @param {type:"boolean"} — TPU only: persist XLA compilation cache across restarts

import os
os.chdir(RUNTIME_DIR)
print("config ok — run cell 3 to launch")

In [ ]:
# @title 3 · Launch worker { display-mode: "form" }
import json, secrets, subprocess, sys, time, urllib.request

TOKEN = AUTH_TOKEN.strip() or secrets.token_hex(16)
open("worker_token.txt", "w").write(TOKEN)

args = [sys.executable, "worker.py", "--tunnel", TUNNEL, "--host", "127.0.0.1",
        "--port", str(PORT), "--token", TOKEN, "--engine", ENGINE, "--device", DEVICE,
        "--attention", ATTENTION, "--attention-hf", ATTENTION_HF]
try:
    import torch_xla  # noqa: F401 — TPU runtime
    if XLA_CACHE:
        args += ["--xla-cache", "/content/rcp_xla_cache"]
except ImportError:
    pass
if NGROK_TOKEN.strip():
    args += ["--ngrok-token", NGROK_TOKEN.strip()]

log_f = open("worker.log", "w")
WORKER = subprocess.Popen(args, stdout=log_f, stderr=subprocess.STDOUT)
print("worker pid:", WORKER.pid, "| log: worker.log")

def _api(path):
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}{path}",
                                 headers={"Authorization": f"Bearer {TOKEN}"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.loads(r.read())

def _fail():
    print(open("worker.log").read()[-3000:])
    raise SystemExit("worker failed to start — log tail above")

for i in range(90):
    time.sleep(2)
    if WORKER.poll() is not None:
        _fail()
    try:
        if _api("/health").get("ok"):
            print(f"worker up after {(i + 1) * 2}s")
            break
    except Exception:
        pass
else:
    _fail()

if TUNNEL != "direct":
    url = None
    for i in range(90):
        time.sleep(2)
        if WORKER.poll() is not None:
            _fail()
        try:
            url = _api("/v1/tunnel").get("public_url")
        except Exception:
            pass
        if url:
            break
    print("#" * 64)
    print("  CONNECT FROM THE LOCAL COMFYUI NODE WITH:")
    print(f"    base_url   : {url or 'TUNNEL NOT READY — check worker.log'}")
    print(f"    auth_token : {TOKEN}")
    print("#" * 64)
else:
    print(f"direct mode — expose port {PORT} yourself | token: {TOKEN}")
print("\nnext: cell 4 to download weights (optional), cell 5 for the control panel")

In [ ]:
# @title 4 · Download a model / LoRA { display-mode: "form" }
import os, re, subprocess

URL = ""  # @param {type:"string"} — direct file URL (e.g. an HF resolve link)
SUBDIR = "text_encoders"  # @param ["text_encoders", "clip", "loras", "embeddings", ""]

assert URL.strip(), "set a URL first"
dest = os.path.join(RUNTIME_DIR, "models", SUBDIR)
os.makedirs(dest, exist_ok=True)
name = os.path.basename(URL.split("?")[0]) or "download.safetensors"
path = os.path.join(dest, name)
subprocess.run(["wget", "-q", "--show-progress", "-O", path, URL.strip()], check=True)
rel = os.path.relpath(path, RUNTIME_DIR).replace("\\", "/")
size_gb = os.path.getsize(path) / 1e9
print(f"saved {rel} ({size_gb:.2f} GB)")
print(f"load it with kind like:  clip_l:{rel}   or via the control panel below")

In [ ]:
# @title 5 · Control Panel { display-mode: "form" }
import json, os, struct, urllib.error, urllib.request
import numpy as np
import torch
import ipywidgets as widgets
from IPython.display import display

PORT = PORT if "PORT" in globals() else 8199
BASE = f"http://127.0.0.1:{PORT}"
TOKEN = open("worker_token.txt").read().strip() if os.path.isfile("worker_token.txt") else ""

NP_DTYPE = {"torch.float16": np.float16, "torch.float32": np.float32,
            "torch.int64": np.int64, "torch.int32": np.int32,
            "torch.uint8": np.uint8, "torch.bool": np.bool_}
TORCH_RESTORE = {"torch.float32": torch.float32, "torch.bfloat16": torch.bfloat16}

def api(method, path, body=None, timeout=600):
    data = json.dumps(body).encode() if isinstance(body, (dict, list)) else body
    req = urllib.request.Request(BASE + path, data=data, method=method,
                                 headers={"Authorization": f"Bearer {TOKEN}",
                                          "Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        raw = r.read()
    try:
        return json.loads(raw)
    except ValueError:
        return raw
def frame(meta):
    meta = dict(meta, tensors={}, blob_size=0, proto=3)
    head = json.dumps(meta).encode()
    return struct.pack(">Q", len(head)) + head

def parse_packed(body):
    n = struct.unpack(">Q", body[:8])[0]
    meta = json.loads(body[8:8 + n])
    blob = body[8 + n:8 + n + meta.get("blob_size", 0)]
    out = {}
    for name, info in meta.get("tensors", {}).items():
        dt = NP_DTYPE.get(info["dtype"])
        assert dt is not None, info["dtype"]
        arr = np.frombuffer(blob[info["offset"]:info["offset"] + info["size"]], dtype=dt)
        t = torch.from_numpy(arr.copy()).reshape(info["shape"])
        restore = TORCH_RESTORE.get(info.get("orig_dtype"))
        if restore is not None and t.dtype != restore:
            t = t.to(restore)
        out[name] = t
    return meta, out

def resolve(obj, tensors):
    if isinstance(obj, dict):
        if set(obj) == {"__tensor__"}:
            return tensors[obj["__tensor__"]]
        return {k: resolve(v, tensors) for k, v in obj.items()}
    if isinstance(obj, list):
        return [resolve(v, tensors) for v in obj]
    return obj

KINDS = ["clip_l", "clip_g", "sdxl", "sd3", "flux", "flux2", "wan", "ltxv",
         "pixart", "chroma", "cosmos", "mochi", "aura_t5", "qwen_image",
         "qwen3vl", "z_image", "minimax_h3", "hunyuan_video", "hydit",
         "lumina2", "anima", "ovis", "omnigen2", "hidream", "native",
         "t5", "sdxl_clip_l", "causal_lm"]

out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px"))

def say(*a):
    with out:
        print(*a)

def guarded(fn):
    def wrap(*_):
        try:
            fn()
        except urllib.error.HTTPError as e:
            say(f"HTTP {e.code}: {e.read().decode('utf-8', 'replace')[:400]}")
        except Exception as e:
            say(f"{type(e).__name__}: {e}")
    return wrap

def refresh():
    s = api("GET", "/v1/status")
    loaded = [m["name"] for m in s.get("models", [])]
    loaded_dd.options = loaded or ["(none)"]
    unload_dd.options = loaded or ["(none)"]
    default_dd.options = loaded or ["(none)"]
    if s.get("default_model") in loaded:
        default_dd.value = s["default_model"]
    say("status | device:", s.get("device"), "| default:", s.get("default_model"),
        "| models:", loaded, "| loras:", len(s.get("loras", [])),
        "| cache:", s.get("cache"))

b_status = widgets.Button(description="Status", button_style="primary")
b_models = widgets.Button(description="Models")
b_loras = widgets.Button(description="LoRAs")
b_tunnel = widgets.Button(description="Tunnel")
b_cache = widgets.Button(description="Clear cache")
b_shutdown = widgets.Button(description="Shutdown", button_style="danger")

@guarded
def on_status(_=None):
    refresh()

@guarded
def on_models(_):
    say(json.dumps(api("GET", "/v1/models"), indent=1)[:1200])

@guarded
def on_loras(_):
    say("loras:", api("GET", "/v1/loras").get("loras"))

@guarded
def on_tunnel(_):
    say(json.dumps(api("GET", "/v1/tunnel"), indent=1))

@guarded
def on_cache(_):
    say("cache cleared:", api("POST", "/v1/cache/clear", {}))

@guarded
def on_shutdown(_):
    api("POST", "/v1/server/shutdown", {"confirm": True})
    say("shutdown requested")

b_status.on_click(on_status); b_models.on_click(on_models); b_loras.on_click(on_loras)
b_tunnel.on_click(on_tunnel); b_cache.on_click(on_cache); b_shutdown.on_click(on_shutdown)

load_name = widgets.Text(placeholder="engine name, e.g. myflux")
load_kind = widgets.Dropdown(options=KINDS, value="clip_l", description="kind")
load_sources = widgets.Text(placeholder="path1.safetensors + path2.safetensors  (native)")
load_source = widgets.Text(placeholder="HF repo or path (single-source kinds)")
load_clip_type = widgets.Text(placeholder="CLIPType for kind=native")
b_load = widgets.Button(description="Load", button_style="success")

@guarded
def on_load(_):
    assert load_name.value.strip(), "set an engine name"
    spec = {"name": load_name.value.strip(), "kind": load_kind.value}
    srcs = [s.strip() for s in load_sources.value.split("+") if s.strip()]
    if srcs:
        spec["sources"] = srcs
    if load_source.value.strip():
        spec["source"] = load_source.value.strip()
    if load_clip_type.value.strip():
        spec["clip_type"] = load_clip_type.value.strip()
    say("loading ... (large checkpoints take a while)")
    say(api("POST", "/v1/models/load", spec))
    refresh()

b_load.on_click(on_load)

default_dd = widgets.Dropdown(description="default")
b_default = widgets.Button(description="Set default", button_style="success")
unload_dd = widgets.Dropdown(description="unload")
b_unload = widgets.Button(description="Unload")

@guarded
def on_default(_):
    say(api("POST", "/v1/models/default", {"name": default_dd.value}))
    refresh()

@guarded
def on_unload(_):
    say(api("POST", "/v1/models/unload", {"name": unload_dd.value}))
    refresh()

b_default.on_click(on_default); b_unload.on_click(on_unload)

encode_text = widgets.Text(value="a photo of a cat", description="encode")
b_encode = widgets.Button(description="Test encode", button_style="info")

@guarded
def on_encode(_):
    body = frame({"text": encode_text.value, "kwargs": {}, "lora_stack": []})
    raw = api("POST", "/v1/encode", body, timeout=1200)
    meta, ts = parse_packed(raw)
    result = resolve(meta["struct"], ts)
    say({k: (tuple(v.shape), str(v.dtype)) if isinstance(v, torch.Tensor) else "present"
         for k, v in result.items()})

b_encode.on_click(on_encode)

panel = widgets.VBox([
    widgets.HTML("<b>Remote CLIP worker — control panel</b>"),
    widgets.HBox([b_status, b_models, b_loras, b_tunnel, b_cache, b_shutdown]),
    widgets.HTML("<b>Load model</b> (native kinds use sources joined by ‘+’; hf kinds use source)"),
    widgets.HBox([load_name, load_kind]),
    load_sources, load_source, load_clip_type, b_load,
    widgets.HTML("<b>Default / unload</b>"),
    widgets.HBox([default_dd, b_default, unload_dd, b_unload]),
    widgets.HTML("<b>Encode test</b> (runs against the default engine)"),
    widgets.HBox([encode_text, b_encode]),
    out,
])
display(panel)
on_status()

In [ ]:
# @title 6 · Worker log tail { display-mode: "form" }
LINES = 40  # @param {type:"integer"}
print("\n".join(open("worker.log").read().splitlines()[-LINES:]))

In [ ]:
# @title 7 · Stop worker { display-mode: "form" }
import subprocess
if "WORKER" in globals() and WORKER.poll() is None:
    WORKER.terminate()
    print("worker terminated (pid", WORKER.pid, ")")
else:
    subprocess.run(["pkill", "-f", "worker.py"])
    print("any stray workers killed")